In [34]:
import numpy as np
import jax 
import jax.numpy as jnp # Enable 64-bit precision (JAX defaults to float32) 5 
jax.config.update("jax_enable_x64", True) 
import scipy

# METHODS

In [35]:
def explicit_euler(t, y, dy_dt, h,args=None,extra_par=None):

    y_new=y+h*dy_dt(y,t, args)

    return y_new

In [36]:
def newton_rhapson(f,f1,x):
    eps=0.5
    while eps+1>1:
        eps=eps/2
    
    count=0
    while f(x)>eps:
        count=count+1
        x=x-f(x)/f1(x)
        if count==4:
            return x
    return x

def f(y,yn,h,dy_dt,tnew):
    
    F=y-yn*h*dy_dt(y,tnew)
    
    return F

def f1(F):
    dF = jax.grad(F) 
    return dF
    

def implicit_euler(t, y, dy_dt, h):

    t_new=t+h
    y_new=newton_rhapson(f,f1,y)

    y_new=y+h*dy_dt(y_new,t_new)

    return y_new

In [71]:
def implicit_euler(t, y, dy_dt, h,args=None,extra_par=None):

    t_new=t+h
    
    y_new=(scipy.optimize.root(lambda x : x-y-h*dy_dt(x,t_new, args), y)).x[0]

    y_new=y+h*dy_dt(y_new,t_new,args)

    return y_new

In [78]:
def explicit_midpoint(t, y, dy_dt, h,args=None,extra_par=None):

    yn=y+h/2*dy_dt(y,t, args)
    ynew=y+h*dy_dt(yn,t+h/2, args)

    return ynew

In [68]:
def Heun(t, y, dy_dt, h,args=None,extra_par=None):

    t_new=t+h
    y_new=y+h*dy_dt(y,t)
    y_new=y+h/2*(dy_dt(y,t)+dy_dt(y_new, t_new,args))

    return y_new

In [72]:
def implicit_midpoint(t, y, dy_dt, h,args=None,extra_par=None):

    t_new=t+h
    
    y_new=(scipy.optimize.root(lambda x : x-y-h*dy_dt(x,t_new,args), y)).x[0]

    y_new=y+h*dy_dt((y+y_new)/2,t+h/2,args)

    return y_new

In [79]:
def RK4(t, y, dy_dt, h,args=None,extra_par=None):

    k1=dy_dt(y,t, args)
    k2=dy_dt(y+h*k1/2,t+h/2, args)
    k3=dy_dt(y+h*k2/2,t+h/2, args)
    k4=dy_dt(y+h*k3,t+h, args)

    y_new=y+h/6*(k1+2*k2+2*k3+k4)

    return y_new 

In [75]:
def AB2(t, y, dy_dt, h,args=None,extra_par=None):
    t_old=t-h
    y_old=extra_par
    y_new=y+h/2*(3*dy_dt(y,t,args)-dy_dt(y_old,t_old,args))

    return y_new

In [22]:
def leapfrog(t, y, dy_dt, h,args,extra_par=None):
    print(y)
    #pos,vel=y
    #mass always first in args
    m=args[0]
    
    dim=len(y)//2
    qold=y[0::2]
    pold=m*y[1::2]
    #print(y)
    print(dy_dt(y,t,args)[1::2])
    pmid=pold+h/2*dy_dt(y,t,args)[1::2]
    #print(pmid)
    qnew=qold+h*pmid/m
    y_fake=np.vstack([qnew,pmid])
    pnew=pmid+h/2*dy_dt(y_fake,t,args)[dim:-1]

    qnew=np.array(qnew)
    pnew=np.array(pnew)
    
    ynew=np.vstack([qnew,pnew])
    #print(ynew)

    return ynew

# INTEGRATOR

In [16]:
def integrate(dy_dt,t_range,IC,h,args=None,method=None,extra_par=None):

    t=t_range[0]
    y=np.array(IC)

    t_array=[t]
    y_array=[y]
    #print(y)
    #print("args",args)

    while t<t_range[1]:

        y_new= method(t,y,dy_dt,h,args,extra_par=extra_par)

        if extra_par!=None:
            extra_par=y#[0]
        
        t+=h
        y=np.array(y_new)

        t_array.append(t)
        y_array.append(y)

        

    t_array,y_array=np.array(t_array),np.array(y_array)
    y_array=y_array.T

    return t_array, y_array